# Setup:

In [1]:
# Import necessary packages
import os
import warnings
warnings.filterwarnings("ignore")
from snowflake.core import Root
from snowflake.core._common import CreateMode
from snowflake.snowpark import Session

In [2]:
# Create session with Root object to manage Snowflake objects
# For VS code 
session = Session.builder.config("connection_name", "myconnection").create()
root = Root(session)

# For Snowsight 
# from snowflake.snowpark.context import get_active_session
# session = get_active_session()

In [3]:
# Create virtual warehouse 
from snowflake.core.warehouse import Warehouse

my_wh = Warehouse(
  name="rag_chatbot_wh",
  warehouse_size="X-SMALL",
  auto_suspend=180,
  auto_resume="true",
  initially_suspended="true"
)
rag_chatbot_wh = root.warehouses.create(my_wh, mode=CreateMode.if_not_exists)

In [4]:
# Create database with schema
from snowflake.core.database import Database
from snowflake.core.schema import Schema

my_db = Database(name="rag_chatbot_db")
rag_chatbot_db = root.databases.create(my_db, mode=CreateMode.if_not_exists)
my_schema = Schema(name="rag_chatbot_schema")
rag_chatbot_schema = root.databases["rag_chatbot_db"].schemas.create(my_schema, mode=CreateMode.if_not_exists)
session.use_schema = rag_chatbot_schema.name

In [5]:
# Create stage for data files
from snowflake.core.stage import Stage, StageDirectoryTable, StageEncryption

my_stage = Stage(
  name="wikipedia_mountain_assets",
  directory_table=StageDirectoryTable(enable=True),
  encryption=StageEncryption(type="SNOWFLAKE_SSE")
)
wikipedia_mountain_assets = root.databases["rag_chatbot_db"].schemas["rag_chatbot_schema"].stages.create(my_stage, mode=CreateMode.if_not_exists)

In [6]:
# Check current session data
print("Role: ", session.get_current_role())
print("Database: ", session.get_current_database())
print("Schema: ", session.get_current_schema())
print("Warehouse: ", session.get_current_warehouse())

Role:  "ACCOUNTADMIN"
Database:  "RAG_CHATBOT_DB"
Schema:  "RAG_CHATBOT_SCHEMA"
Warehouse "RAG_CHATBOT_WH"


# Ingest data to Snowflake:

In [7]:
# Metadata
import pandas as pd
file_url_info = pd.read_csv(f"{os.getcwd()}\\assets\\file_url_info.csv")
file_url_info.columns = file_url_info.columns.str.upper()
file_url_info.head()

,FILE_NAME,URL_LINK
0,Mountain.pdf,https://en.wikipedia.org/wiki/Mountain
1,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...


In [8]:
# Ingest metadata to Snowflake database
session.write_pandas(df=file_url_info, 
                     table_name="FILE_URL_INFO",
                     auto_create_table=True,
                     overwrite=True)
print("FILE_URL_INFO Table Successfully Created!")

FILE_URL_INFO Table Successfully Created!


In [ ]:
#from snowflake.core.table import Table, TableColumn, PrimaryKey, UniqueKey
#
#my_table = Table(
#  name="file_url_info",
#  columns=[TableColumn(name="file_name", datatype="string", constraints=[PrimaryKey()]),
#           TableColumn(name="url_link", datatype="string", constraints=[UniqueKey()])]
#)
#file_url_info = root.databases["rag_chatbot_db"].schemas["rag_chatbot_schema"].tables.create(my_table)

In [9]:
# Ingest data to Snowflake Internal Stage
for file_name in file_url_info["FILE_NAME"]:
    session.file.put(
        local_file_name=f"{os.getcwd()}\\assets\\{file_name}",
        stage_location="@wikipedia_mountain_assets",
        overwrite=True,
        auto_compress=False)
print("Files Successfully Uploaded to Snowflake Stage!")

Files Successfully Uploaded to Snowflake Stage!


In [13]:
# Register UDTF for data flow from Snowflake Internal Stage to Vector Database
from snowflake.snowpark.types import StringType, StructField, StructType
from snowflake.snowpark.functions import udtf
from langchain.text_splitter import RecursiveCharacterTextSplitter
from snowflake.snowpark.files import SnowflakeFile
import PyPDF2, io
import pandas as pd

@udtf(session=session,
      output_schema=StructType([StructField("chunk", StringType())]),
      input_types=[StringType()],
      name="pdf_text_chunker",
      is_permanent=True,
      stage_location="@wikipedia_mountain_assets",
      replace=True,
      packages=["langchain==0.1.15", 
                "PyPDF2==2.10.5", 
                "snowflake-ml-python==1.6.0"]
      )
class pdf_text_chunker:

    def read_pdf(self, file_url):
    
        with SnowflakeFile.open(file_url, 'rb') as f:
            buffer = io.BytesIO(f.readall())
            
        reader = PyPDF2.PdfReader(buffer)   
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        return text

    def process(self, file_url):

        text = self.read_pdf(file_url)
        
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = 4000, 
            chunk_overlap  = 400, 
            length_function = len
        )
    
        chunks = text_splitter.split_text(text)
        df = pd.DataFrame(chunks, columns=['chunks'])
        yield from df.itertuples(index=False, name=None)

print("UDFT Successfully Registered!")

In [14]:
# Create and insert data to vector database
sql_query = """create or replace table vector_db as
               select file_name, url_link, func.chunk as chunk, SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2',chunk) as chunk_vec 
               from file_url_info, TABLE(pdf_text_chunker(build_scoped_file_url(@wikipedia_mountain_assets, file_name ))) as func;"""

session.sql(sql_query).collect()
print("VECTOR_DB Table Successfully Created!")

VECTOR_DB Table Successfully Created!


In [15]:
# View VECTOR_DB Table data
session.table("vector_db").to_pandas().head(5)

,FILE_NAME,URL_LINK,CHUNK,CHUNK_VEC
0,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...,Physiographic world map with mountain ranges a...,"[-0.023530355, 0.007084886, -0.04706224, -0.00..."
1,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...,"1. High Himalayas – 2,300 km (1,400 mi)[6]\n2....","[-0.018259777, 0.0032187363, -0.027193319, -0...."
2,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...,55. Precordillera – 800 km (500 mi) (considere...,"[-0.025266536, -0.0018097094, -0.038225185, 0...."
3,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...,"32. Kirthar Mountains, Pakistan\n33. Knuckles ...","[-0.018622803, -0.01292687, -0.041508134, -0.0..."
4,List_of_mountain_ranges.pdf,https://en.wikipedia.org/wiki/List_of_mountain...,"2. Berkovska Mountain, Bulgaria[11]\n3. Vracha...","[0.021784097, -0.00023464879, -0.03756812, 0.0..."


In [ ]:
#from snowflake.core.table import Table, TableColumn
#
#my_table = Table(
#  name="vector_db",
#  columns=[TableColumn(name="file_name", datatype="string"),
#           TableColumn(name="url_link", datatype="string"),
#           TableColumn(name="chunk", datatype="string"),
#           TableColumn(name="chunk_vec", datatype="vector")] --> VECTOR DATATYPE STRING CONSTRAINT NOT ADDED YET
#)
#vector_db = root.databases["rag_chatbot_db"].schemas["rag_chatbot_schema"].tables.create(my_table, mode=CreateMode.if_not_exists)

# Test RAG setup:

In [16]:
#from snowflake.snowpark.functions import vector_cosine_distance
from snowflake.cortex import Complete

In [17]:
# Semantic search for chunks based on user query
sql_query = """
            with results as 
            (select url_link, VECTOR_COSINE_SIMILARITY(vector_db.chunk_vec,
            SNOWFLAKE.CORTEX.EMBED_TEXT_768('e5-base-v2', ?)) as similarity, chunk
            from vector_db
            order by similarity desc
            limit ?)
            select chunk, similarity, url_link from results 
            """
myquestion = "What is a mountain?"
num_chunks = 3
df_context = session.sql(sql_query, params=[myquestion, num_chunks]).to_pandas() 
df_context

,CHUNK,SIMILARITY,URL_LINK
0,"Mount Everest, Earth's highest mountain\nMount...",0.814696,https://en.wikipedia.org/wiki/Mountain
1,from the original on 8 February 2013. Retrieve...,0.804810,https://en.wikipedia.org/wiki/Mountain
2,"3,000 metres (9,800 ft) elevation.[49] About h...",0.796534,https://en.wikipedia.org/wiki/Mountain


In [18]:
# Merging all chunks into a single context
context_length =  len(df_context) -1
prompt_context = ""
for i in range (0, context_length):
    prompt_context += df_context._get_value(i, 'CHUNK')

In [19]:
# Call Snowflake cortex Complete function on user query

model = "llama3-8b"
prompt = f"""
          You are an expert assistant. Extract information from context provided. 
          Answer the question based on the context. 
          Be concise and do not hallucinate. 
          If you do not have the information just say so.
          Do not mention the CONTEXT used in your answer.
          Do not add answer tag in your answer.
          Context: {prompt_context}
          Question:  
          {myquestion} 
          Answer: 
          """

answer = Complete(model=model, prompt=prompt)
print(answer)

A mountain is a natural elevation of the earth's surface rising more or less abruptly from the surrounding level and attaining an altitude which, relatively to the adjacent elevation, is impressive or notable.
